# XGBoost Regression

training xgboost regression in blocks using baseline features + traficom features


## 1. Imports

This notebook keeps the same overall workflow as the linear-regression and random-forest notebooks, but uses native-categorical XGBoost with validation-based early stopping and tuning on the recommended feature set.


In [148]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


## 2. Load data

In [149]:
train_path = "../../datasets/splits/train_grouped.csv"
validation_path = "../../datasets/splits/validation_grouped.csv"

In [150]:
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

for dataset_df in [train_df, validation_df]:
    for date_column in ["first_seen_date", "last_seen_date", "scrape_date"]:
        if date_column in dataset_df.columns:
            dataset_df[date_column] = pd.to_datetime(dataset_df[date_column], errors="coerce")

reference_first_seen_date = min(
    df["first_seen_date"].min()
    for df in [train_df, validation_df]
    if "first_seen_date" in df.columns
)

for dataset_df in [train_df, validation_df]:
    dataset_df["first_seen_day_offset"] = (
        dataset_df["first_seen_date"] - reference_first_seen_date
    ).dt.days
    dataset_df["last_seen_day_offset"] = (
        dataset_df["last_seen_date"] - reference_first_seen_date
    ).dt.days
    dataset_df["listing_midpoint_day_offset"] = (
        dataset_df["first_seen_day_offset"]
        + dataset_df["last_seen_day_offset"]
    ) / 2


## 3. Quick data checks

In [151]:
print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (7954, 82)
Validation shape: (1689, 82)


In [152]:
print("Train info")
train_df.info()

print("\nValidation info")
validation_df.info()

Train info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7954 entries, 0 to 7953
Data columns (total 82 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   product_id                      7954 non-null   int64         
 1   part_name                       7954 non-null   object        
 2   price                           7954 non-null   float64       
 3   quality_grade                   7954 non-null   object        
 4   oem_number                      7954 non-null   object        
 5   mileage                         7954 non-null   float64       
 6   brand                           7954 non-null   object        
 7   model                           7954 non-null   object        
 8   category                        7954 non-null   object        
 9   subcategory                     7954 non-null   object        
 10  scrape_date                     7954 non-null   datetime64[ns

## 4. Define feature groups

Registry lifecycle features describe vehicle registration-history patterns from Traficom data. Listing dynamics features describe scrape-window behavior of listings.

In [153]:
baseline_features = [
    "part_name",
    "quality_grade",
    "oem_number",
    "mileage",
    "brand",
    "model",
    "category",
    "subcategory",
    "year_start",
    "year_end",
    "year_span",
    "year_mid",
    "repair_status",
]

traficom_features = [
    "model_total_registered",
    "model_median_vehicle_age",
    "model_mean_vehicle_age",
    "model_median_mileage",
    "model_mean_mileage",
    "model_median_engine_cc",
    "model_median_power_kw",
    "model_median_mass_kg",
    "brand_total_registered",
    "brand_median_vehicle_age",
    "brand_mean_vehicle_age",
    "brand_median_mileage",
    "brand_mean_mileage",
]

# Registry lifecycle features describe vehicle registration-history features.
registry_lifecycle_candidates = [
    "model_firstreg_total_2014_2026",
    "model_firstreg_recent_share",
    "model_firstreg_old_share",
    "model_firstreg_weighted_year",
    "model_firstreg_peak_year",
    "model_firstreg_peak_count",
    "model_firstreg_year_span",
    "brand_firstreg_total_2014_2026",
    "brand_firstreg_recent_share",
    "brand_firstreg_old_share",
    "brand_firstreg_weighted_year",
    "brand_firstreg_peak_year",
    "brand_firstreg_peak_count",
    "brand_firstreg_year_span",
]

registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates 
    if column in train_df.columns

]

missing_registry_lifecycle_features = [
    column for column in registry_lifecycle_candidates 
    if column not in train_df.columns
]

traficom_extended_candidates = [
    "model_share_of_market",
    "model_share_within_brand",
    "model_share_over_10y",
    "model_share_over_200k_km",
    "model_automatic_share",
    "model_petrol_share",
    "model_diesel_share",
    "model_ev_share",
    "model_hybrid_share",
    "model_turbo_share",
    "brand_median_engine_cc",
    "brand_median_power_kw",
    "brand_median_mass_kg",
    "brand_share_of_market",
    "brand_share_over_10y",
    "brand_share_over_200k_km",
    "brand_automatic_share",
    "brand_petrol_share",
    "brand_diesel_share",
    "brand_ev_share",
    "brand_hybrid_share",
    "brand_turbo_share",
]

traficom_extended_features = [
    column for column in traficom_extended_candidates if column in train_df.columns
]

missing_traficom_extended_features = [
    column for column in traficom_extended_candidates if column not in train_df.columns
]

print("Found registry lifecycle features:")
print(registry_lifecycle_features)

print("\nMissing registry lifecycle features:")
print(missing_registry_lifecycle_features)

print("\nFound extended Traficom features:")
print(traficom_extended_features)

print("\nMissing extended Traficom features:")
print(missing_traficom_extended_features)

# Listing dynamics features describe scrape-window listing behavior,
# not registry lifecycle or vehicle registration-history features.
listing_dynamics_features = [
    "times_observed",
    "observed_span_days",
    "price_changed_flag",
    "price_change_count",
    "absolute_price_change",
    "relative_price_change_pct",
    "first_seen_day_offset",
    "last_seen_day_offset",
    "listing_midpoint_day_offset",
]

# Keep this alias so the existing notebook cells continue to run.
lifecycle_features = listing_dynamics_features

recommended_inclusion_reasons = {
    "first_seen_day_offset": "Numeric position of listing start within the crawl window for XGBoost.",
    "last_seen_day_offset": "Numeric position of listing end within the crawl window for XGBoost.",
    "listing_midpoint_day_offset": "Numeric midpoint of listing visibility window for XGBoost.",
    "brand_is_known_model_family": "Low-cardinality metadata flag; slight validation improvement when included.",
    "mileage_missing_flag": "Tracks rows where mileage was originally missing before hierarchical median imputation.",
}

recommended_exclusion_reasons = {
    "product_id": "High-cardinality listing identifier; encourages memorization and hurt MAE.",
    "scrape_date": "Current crawl wave rather than part value; worsened validation when included.",
    "brand_merge_key": "Redundant normalized brand key that overlaps with brand.",
    "model_merge_key": "Redundant normalized model key that overlaps with model.",
    "model_family_clean": "Collapsed model family duplicate of the existing model labels.",
    "model_looks_like_part_taxonomy": "Constant in the training split, so it adds no signal.",
}

manual_all_feature_groups = list(dict.fromkeys(
    baseline_features
    + traficom_features
    + registry_lifecycle_features
    + traficom_extended_features
    + listing_dynamics_features
))

xgboost_specific_exclusion_reasons = {
    "first_seen_date": "Raw date string replaced with numeric day-offset feature for XGBoost.",
    "last_seen_date": "Raw date string replaced with numeric day-offset feature for XGBoost.",
    "part_name": "High-cardinality part label made XGBoost sparse and unstable; subcategory remains available.",
}

recommended_excluded_features = set(recommended_exclusion_reasons)
xgboost_excluded_features = recommended_excluded_features | set(xgboost_specific_exclusion_reasons)

recommended_model_features = [
    column for column in train_df.columns
    if column != "price" and column not in xgboost_excluded_features
]

missing_from_previous_manual_all = [
    column for column in recommended_model_features if column not in manual_all_feature_groups
]

feature_audit_df = pd.DataFrame(
    [
        {
            "feature": column,
            "status": "missing_from_previous_manual_all",
            "reason": recommended_inclusion_reasons.get(
                column,
                "New candidate feature found in the dataset and included in the recommended model feature set.",
            ),
        }
        for column in missing_from_previous_manual_all
    ]
    + [
        {
            "feature": column,
            "status": "recommended_exclusion",
            "reason": recommended_exclusion_reasons[column],
        }
        for column in sorted(recommended_excluded_features)
    ]
    + [
        {
            "feature": column,
            "status": "xgboost_specific_exclusion",
            "reason": xgboost_specific_exclusion_reasons[column],
        }
        for column in sorted(xgboost_specific_exclusion_reasons)
    ]
)

print("Columns missing from the previous manual all-features set:")
print(missing_from_previous_manual_all)

print("\nRecommended exclusions:")
print(sorted(recommended_excluded_features))

print("\nXGBoost-specific exclusions:")
print(sorted(xgboost_specific_exclusion_reasons))

print("\nNumber of recommended model features:", len(recommended_model_features))

feature_audit_df

Found registry lifecycle features:
['model_firstreg_total_2014_2026', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_year_span', 'brand_firstreg_total_2014_2026', 'brand_firstreg_recent_share', 'brand_firstreg_old_share', 'brand_firstreg_weighted_year', 'brand_firstreg_peak_year', 'brand_firstreg_peak_count', 'brand_firstreg_year_span']

Missing registry lifecycle features:
[]

Found extended Traficom features:
['model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over_10y', 'brand_share_over_200k_km', 'brand_automatic_share', 'brand_petrol_share', 'brand_diesel_share', 'brand_ev_s

,feature,status,reason
0,brand_is_known_model_family,missing_from_previous_manual_all,Low-cardinality metadata flag; slight validati...
1,mileage_missing_flag,missing_from_previous_manual_all,Tracks rows where mileage was originally missi...
2,brand_merge_key,recommended_exclusion,Redundant normalized brand key that overlaps w...
3,model_family_clean,recommended_exclusion,Collapsed model family duplicate of the existi...
4,model_looks_like_part_taxonomy,recommended_exclusion,"Constant in the training split, so it adds no ..."
5,model_merge_key,recommended_exclusion,Redundant normalized model key that overlaps w...
6,product_id,recommended_exclusion,High-cardinality listing identifier; encourage...
7,scrape_date,recommended_exclusion,Current crawl wave rather than part value; wor...
8,first_seen_date,xgboost_specific_exclusion,Raw date string replaced with numeric day-offs...
9,last_seen_date,xgboost_specific_exclusion,Raw date string replaced with numeric day-offs...


## 5. Select baseline feature set

In [154]:
features_baseline_only = list(dict.fromkeys(
    [
        feature for feature in baseline_features
        if feature not in xgboost_excluded_features
    ]
    + [
        "first_seen_day_offset",
        "last_seen_day_offset",
        "listing_midpoint_day_offset",
    ]
))

assert len(features_baseline_only) > 0, "Add at least one column to baseline_features before training."

print("Number of baseline features:", len(features_baseline_only))
features_baseline_only

Number of baseline features: 15


['quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'first_seen_day_offset',
 'last_seen_day_offset',
 'listing_midpoint_day_offset']

## 6. Build X and y

In [155]:
missing_features = [column for column in features_baseline_only if column not in train_df.columns]
assert len(missing_features) == 0, f"These selected features are missing from the dataset: {missing_features}"

X_train = train_df[features_baseline_only].copy()
X_validation = validation_df[features_baseline_only].copy()

y_train = train_df["price"].copy()
y_validation = validation_df["price"].copy()

## 7. Log-transform target

In [156]:
y_train_log = np.log(y_train)
y_validation_log = np.log(y_validation)

y_train_log.head()


0    5.179534
1    5.179534
2    5.179534
3    5.179534
4    3.165475
Name: price, dtype: float64

## 8. Detect column types

In [157]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features

['mileage',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'first_seen_day_offset',
 'last_seen_day_offset',
 'listing_midpoint_day_offset']

## 8. Detect column types

In [158]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

In [159]:
if "numeric_features" not in globals() or "categorical_features" not in globals():
    numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

column_type_summary = pd.DataFrame(
    {
        "column_type": ["numeric", "categorical"],
        "count": [len(numeric_features), len(categorical_features)],
        "columns": [numeric_features, categorical_features],
    }
)

display(column_type_summary)

,column_type,count,columns
0,numeric,8,"[mileage, year_start, year_end, year_span, yea..."
1,categorical,7,"[quality_grade, oem_number, brand, model, cate..."


## 9. Tune Native-Categorical XGBoost


In [160]:
xgboost_candidate_params = {
    "robust_categorical": {
        "objective": "reg:pseudohubererror",
        "eval_metric": "rmse",
        "n_estimators": 3000,
        "learning_rate": 0.02,
        "max_depth": 4,
        "min_child_weight": 10,
        "gamma": 0.2,
        "subsample": 0.75,
        "colsample_bytree": 0.6,
        "colsample_bylevel": 0.7,
        "reg_alpha": 0.5,
        "reg_lambda": 5.0,
        "max_bin": 256,
        "max_cat_to_onehot": 8,
        "max_cat_threshold": 64,
    },
    "balanced_categorical": {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "n_estimators": 2500,
        "learning_rate": 0.03,
        "max_depth": 5,
        "min_child_weight": 8,
        "gamma": 0.1,
        "subsample": 0.8,
        "colsample_bytree": 0.7,
        "colsample_bylevel": 0.8,
        "reg_alpha": 0.2,
        "reg_lambda": 3.0,
        "max_bin": 256,
        "max_cat_to_onehot": 8,
        "max_cat_threshold": 64,
    },
    "very_conservative": {
        "objective": "reg:pseudohubererror",
        "eval_metric": "rmse",
        "n_estimators": 4000,
        "learning_rate": 0.015,
        "max_depth": 3,
        "min_child_weight": 12,
        "gamma": 0.3,
        "subsample": 0.7,
        "colsample_bytree": 0.55,
        "colsample_bylevel": 0.7,
        "reg_alpha": 1.0,
        "reg_lambda": 8.0,
        "max_bin": 256,
        "max_cat_to_onehot": 4,
        "max_cat_threshold": 32,
    },
}


def align_xgboost_frames(X_train_current, X_validation_current):
    X_train_prepared_current = X_train_current.copy()
    X_validation_prepared_current = X_validation_current.copy()

    datetime_columns = X_train_prepared_current.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns.tolist()
    if len(datetime_columns) > 0:
        X_train_prepared_current = X_train_prepared_current.drop(columns=datetime_columns)
        X_validation_prepared_current = X_validation_prepared_current.drop(columns=datetime_columns, errors="ignore")

    categorical_columns = X_train_prepared_current.select_dtypes(include=["object", "category"]).columns.tolist()
    for column in categorical_columns:
        train_as_string = X_train_prepared_current[column].astype("string")
        validation_as_string = X_validation_prepared_current[column].astype("string")

        categories = pd.Index(train_as_string.dropna().unique())
        if len(categories) == 0:
            categories = pd.Index(["__missing__"])

        X_train_prepared_current[column] = pd.Categorical(train_as_string, categories=categories)
        X_validation_prepared_current[column] = pd.Categorical(validation_as_string, categories=categories)

    return X_train_prepared_current, X_validation_prepared_current


def make_xgboost_regressor(params=None):
    model_params = {
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": 42,
        "n_jobs": -1,
        "early_stopping_rounds": 100,
    }
    if params is not None:
        model_params.update(params)

    return XGBRegressor(**model_params)


def fit_xgboost_model(X_train_current, y_train_log_current, X_validation_current, y_validation_log_current, params):
    X_train_prepared_current, X_validation_prepared_current = align_xgboost_frames(
        X_train_current,
        X_validation_current,
    )

    model_current = make_xgboost_regressor(params)
    model_current.fit(
        X_train_prepared_current,
        y_train_log_current,
        eval_set=[(X_validation_prepared_current, y_validation_log_current)],
        verbose=False,
    )

    return model_current, X_train_prepared_current, X_validation_prepared_current


In [161]:
tuning_features = recommended_model_features

X_train_tuning = train_df[tuning_features].copy()
X_validation_tuning = validation_df[tuning_features].copy()

tuning_results = []
for config_name, config_params in xgboost_candidate_params.items():
    model_current, X_train_tuning_prepared, X_validation_tuning_prepared = fit_xgboost_model(
        X_train_tuning,
        y_train_log,
        X_validation_tuning,
        y_validation_log,
        config_params,
    )

    validation_pred_log_current = model_current.predict(X_validation_tuning_prepared)
    validation_pred_current = np.exp(validation_pred_log_current)

    tuning_results.append({
        "config": config_name,
        "validation_MAE": mean_absolute_error(y_validation, validation_pred_current),
        "validation_RMSE": np.sqrt(mean_squared_error(y_validation, validation_pred_current)),
        "validation_R2": r2_score(y_validation, validation_pred_current),
        "best_iteration": getattr(model_current, "best_iteration", None),
    })

tuning_results_df = pd.DataFrame(tuning_results).sort_values(
    ["validation_RMSE", "validation_MAE"]
).reset_index(drop=True)

selected_xgboost_config_name = tuning_results_df.iloc[0]["config"]
selected_xgboost_params = xgboost_candidate_params[selected_xgboost_config_name]

print("Selected XGBoost config:", selected_xgboost_config_name)
print(selected_xgboost_params)

display(
    tuning_results_df.style.format({
        "validation_MAE": "{:.4f}",
        "validation_RMSE": "{:.4f}",
        "validation_R2": "{:.4f}",
    })
)


Selected XGBoost config: balanced_categorical
{'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'n_estimators': 2500, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 8, 'gamma': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.7, 'colsample_bylevel': 0.8, 'reg_alpha': 0.2, 'reg_lambda': 3.0, 'max_bin': 256, 'max_cat_to_onehot': 8, 'max_cat_threshold': 64}


,config,validation_MAE,validation_RMSE,validation_R2,best_iteration
0,balanced_categorical,17.5436,45.9620,0.9934,1261
1,robust_categorical,26326189905661000197921886151693565952.0000,26326189905661000197921886151693565952.0000,-2155918697518685039303665842892606457926461381427547637903330834907136.0000,0
2,very_conservative,26326189905661000197921886151693565952.0000,26326189905661000197921886151693565952.0000,-2155918697518685039303665842892606457926461381427547637903330834907136.0000,0


## 10. Train tuned baseline XGBoost


In [162]:
baseline_model, X_train_prepared, X_validation_prepared = fit_xgboost_model(
    X_train,
    y_train_log,
    X_validation,
    y_validation_log,
    selected_xgboost_params,
)


## 11. Predict and convert back to euro scale

In [163]:
validation_pred_log = baseline_model.predict(X_validation_prepared)


In [164]:
validation_pred = np.exp(validation_pred_log)

## 12. Preview Baseline XGBoost Predictions


In [165]:
baseline_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred,
})

baseline_prediction_preview.head()

,actual_price,predicted_price
0,177.6,176.452942
1,473.6,428.243286
2,142.1,162.723267
3,82.9,94.369446
4,177.6,145.942322


## 13. Baseline + Traficom features

This section tests whether external Traficom enrichment improves prediction compared to the listing-only baseline.

In [166]:
features_baseline_plus_traficom = list(dict.fromkeys(
    features_baseline_only + traficom_features
))

print("Number of baseline + Traficom features:", len(features_baseline_plus_traficom))
features_baseline_plus_traficom

Number of baseline + Traficom features: 28


['quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'first_seen_day_offset',
 'last_seen_day_offset',
 'listing_midpoint_day_offset',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_mean_mileage']

## 14. Build X and y for baseline + Traficom

In [167]:
missing_traficom_features = [
    column for column in features_baseline_plus_traficom if column not in train_df.columns
]
assert len(missing_traficom_features) == 0, (
    f"These selected features are missing from the dataset: {missing_traficom_features}"
)

X_train_traficom = train_df[features_baseline_plus_traficom].copy()
X_validation_traficom = validation_df[features_baseline_plus_traficom].copy()

## 15. Detect column types again

In [168]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()

In [169]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'first_seen_day_offset', 'last_seen_day_offset', 'listing_midpoint_day_offset', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 16. Prepare and train baseline + Traficom XGBoost


In [170]:
numeric_features_traficom = X_train_traficom.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_traficom = X_train_traficom.select_dtypes(include=["object", "category"]).columns.tolist()


In [171]:
print("Numeric columns:", numeric_features_traficom)
print("\nCategorical columns:", categorical_features_traficom)

xgboost_traficom, X_train_traficom_prepared, X_validation_traficom_prepared = fit_xgboost_model(
    X_train_traficom,
    y_train_log,
    X_validation_traficom,
    y_validation_log,
    selected_xgboost_params,
)


Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'first_seen_day_offset', 'last_seen_day_offset', 'listing_midpoint_day_offset', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage']

Categorical columns: ['quality_grade', 'oem_number', 'brand', 'model', 'category', 'subcategory', 'repair_status']


## 17. Predict baseline + Traficom XGBoost


In [172]:
validation_pred_log_traficom = xgboost_traficom.predict(X_validation_traficom_prepared)


## 18. Predict on validation and convert back to euro scale

In [173]:
validation_pred_traficom = np.exp(validation_pred_log_traficom)


In [174]:
validation_pred_traficom = np.exp(validation_pred_log_traficom)

## 19. Preview Baseline + Traficom XGBoost Predictions


In [175]:
traficom_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_traficom,
})

traficom_prediction_preview.head()

,actual_price,predicted_price
0,177.6,179.894547
1,473.6,408.547852
2,142.1,154.183762
3,82.9,91.740265
4,177.6,144.503082


In [176]:
traficom_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,252.462845
std,567.153248,557.194702
min,5.900000,9.595146
25%,59.000000,57.373882
50%,100.600000,93.423523
75%,236.000000,223.605209
max,4284.000000,4115.369629


## 20. All Recommended Features

This section trains the XGBoost model on every recommended feature: the full manual feature union plus `first_seen_date`, `last_seen_date`, and `brand_is_known_model_family`, while still excluding IDs, duplicate keys, and the constant taxonomy flag.


In [177]:
features_all = recommended_model_features

print("Number of recommended model features:", len(features_all))
features_all

Number of recommended model features: 72


['quality_grade',
 'oem_number',
 'mileage',
 'brand',
 'model',
 'category',
 'subcategory',
 'year_start',
 'year_end',
 'year_span',
 'year_mid',
 'repair_status',
 'brand_is_known_model_family',
 'model_total_registered',
 'model_median_vehicle_age',
 'model_mean_vehicle_age',
 'model_median_mileage',
 'model_mean_mileage',
 'model_median_engine_cc',
 'model_median_power_kw',
 'model_median_mass_kg',
 'model_share_of_market',
 'model_share_within_brand',
 'model_share_over_10y',
 'model_share_over_200k_km',
 'model_automatic_share',
 'model_petrol_share',
 'model_diesel_share',
 'model_ev_share',
 'model_hybrid_share',
 'model_turbo_share',
 'model_firstreg_total_2014_2026',
 'model_firstreg_year_span',
 'model_firstreg_peak_year',
 'model_firstreg_peak_count',
 'model_firstreg_recent_share',
 'model_firstreg_old_share',
 'model_firstreg_weighted_year',
 'brand_total_registered',
 'brand_median_vehicle_age',
 'brand_mean_vehicle_age',
 'brand_median_mileage',
 'brand_mean_mileage',

## 21. Build X and y for all recommended features

In [178]:
missing_all_features = [column for column in features_all if column not in train_df.columns]
assert len(missing_all_features) == 0, (
    f"These selected features are missing from the dataset: {missing_all_features}"
)

X_train_all = train_df[features_all].copy()
X_validation_all = validation_df[features_all].copy()

## 22. Detect column types again

In [179]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()

In [180]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 23. Prepare and train recommended-feature XGBoost


In [181]:
numeric_features_all = X_train_all.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features_all = X_train_all.select_dtypes(include=["object", "category"]).columns.tolist()


In [182]:
print("Numeric columns:", numeric_features_all)
print("\nCategorical columns:", categorical_features_all)

xgboost_all, X_train_all_prepared, X_validation_all_prepared = fit_xgboost_model(
    X_train_all,
    y_train_log,
    X_validation_all,
    y_validation_log,
    selected_xgboost_params,
)


Numeric columns: ['mileage', 'year_start', 'year_end', 'year_span', 'year_mid', 'brand_is_known_model_family', 'model_total_registered', 'model_median_vehicle_age', 'model_mean_vehicle_age', 'model_median_mileage', 'model_mean_mileage', 'model_median_engine_cc', 'model_median_power_kw', 'model_median_mass_kg', 'model_share_of_market', 'model_share_within_brand', 'model_share_over_10y', 'model_share_over_200k_km', 'model_automatic_share', 'model_petrol_share', 'model_diesel_share', 'model_ev_share', 'model_hybrid_share', 'model_turbo_share', 'model_firstreg_total_2014_2026', 'model_firstreg_year_span', 'model_firstreg_peak_year', 'model_firstreg_peak_count', 'model_firstreg_recent_share', 'model_firstreg_old_share', 'model_firstreg_weighted_year', 'brand_total_registered', 'brand_median_vehicle_age', 'brand_mean_vehicle_age', 'brand_median_mileage', 'brand_mean_mileage', 'brand_median_engine_cc', 'brand_median_power_kw', 'brand_median_mass_kg', 'brand_share_of_market', 'brand_share_over

## 24. Predict recommended-feature XGBoost


In [183]:
validation_pred_log_all = xgboost_all.predict(X_validation_all_prepared)


## 25. Predict on validation and convert back to euro scale

In [184]:
validation_pred_all = np.exp(validation_pred_log_all)


In [185]:
validation_pred_all = np.exp(validation_pred_log_all)

## 26. Preview Recommended-Feature Predictions

These cells only preview the validation predictions in euro scale for the recommended-feature model. Formal model-comparison metrics and grouped diagnostics are reported in the later validation sections.

In [186]:
# Pair each validation observation with its prediction in euro scale.
recommended_prediction_preview = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": validation_pred_all,
})

recommended_prediction_preview.head()

,actual_price,predicted_price
0,177.6,180.899185
1,473.6,459.759674
2,142.1,148.297852
3,82.9,82.722885
4,177.6,172.708405


In [187]:
# A quick descriptive summary helps check the prediction range before detailed validation.
recommended_prediction_preview.describe()

,actual_price,predicted_price
count,1689.000000,1689.000000
mean,258.793428,255.766357
std,567.153248,559.416992
min,5.900000,7.660582
25%,59.000000,54.986919
50%,100.600000,102.367722
75%,236.000000,229.492783
max,4284.000000,4288.511719


## 27. Validation Error Analysis By Part Group

This section evaluates validation-set errors for the existing XGBoost model by part grouping. It reuses the available validation predictions and reports which groups are predicted most accurately and least accurately.


In [188]:
# Reuse the most detailed validation predictions already available in the notebook.
if "best_validation_predictions" in globals():
    validation_predictions_eur = best_validation_predictions
elif "validation_pred_all" in globals():
    validation_predictions_eur = validation_pred_all
elif "validation_pred_traficom" in globals():
    validation_predictions_eur = validation_pred_traficom
elif "validation_pred" in globals():
    validation_predictions_eur = validation_pred
else:
    raise NameError("No validation predictions were found in the notebook.")

# Build one validation results table in euro scale for grouped error analysis.
validation_results_df = validation_df[["price", "category", "subcategory", "part_name"]].copy()
validation_results_df = validation_results_df.rename(columns={"price": "actual_price"})
validation_results_df["predicted_price"] = validation_predictions_eur
validation_results_df["absolute_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
).abs()
validation_results_df["squared_error"] = (
    validation_results_df["actual_price"]
    - validation_results_df["predicted_price"]
) ** 2

validation_results_df.head()

,actual_price,category,subcategory,part_name,predicted_price,absolute_error,squared_error
0,177.6,airbag,contact roll airbag,"Contact roll Airbag - , e-",186.123413,8.523413,72.648571
1,473.6,airbag,passenger airbag,"Passenger airbag - , e-",470.261200,3.338800,11.147586
2,142.1,airbag,left,"Curtain airbags - , e-(Left)",144.611938,2.511938,6.309835
3,82.9,airbag,seat assembled,"Seat assembled - , e-(Right front)",81.395897,1.504103,2.262326
4,177.6,airbag,all,"Side airbags - , e-(Right)",181.995560,4.395560,19.320945


In [189]:
def summarize_group_errors(validation_results_df, group_column, min_count=20):
    summary = (
        validation_results_df
        .groupby(group_column, observed=False)
        .agg(
            count=("actual_price", "size"),
            MAE=("absolute_error", "mean"),
            median_absolute_error=("absolute_error", "median"),
            RMSE=("squared_error", lambda s: np.sqrt(s.mean())),
            within_10_eur=("absolute_error", lambda s: (s <= 10).mean()),
            within_20_eur=("absolute_error", lambda s: (s <= 20).mean()),
            within_50_eur=("absolute_error", lambda s: (s <= 50).mean()),
        )
        .reset_index()
    )

    summary = summary[summary["count"] >= min_count].copy()
    return summary.sort_values(["MAE", "RMSE", "count"]).reset_index(drop=True)


category_error_summary = summarize_group_errors(validation_results_df, "category")
subcategory_error_summary = summarize_group_errors(validation_results_df, "subcategory")
part_name_error_summary = summarize_group_errors(validation_results_df, "part_name")

## 28. Best And Worst Groups By MAE

The tables below summarize grouped validation errors in absolute euro terms. The `within_10_eur`, `within_20_eur`, and `within_50_eur` columns show how often predictions fall within practical absolute-error thresholds.

In [190]:
# Show the main absolute-error metrics used for grouped validation review.
summary_display_columns = [
    "count",
    "MAE",
    "median_absolute_error",
    "RMSE",
    "within_10_eur",
    "within_20_eur",
    "within_50_eur",
]

for summary_name, summary_df, group_label in [
    ("Category", category_error_summary, "category"),
    ("Subcategory", subcategory_error_summary, "subcategory"),
    ("Part name", part_name_error_summary, "part_name"),
]:
    print(f"\n{summary_name}: 10 best groups by MAE")
    display(
        summary_df.head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )

    print(f"{summary_name}: 10 worst groups by MAE")
    display(
        summary_df.sort_values(["MAE", "RMSE", "count"], ascending=[False, False, False]).head(10).style.format({
            "MAE": "{:.2f}",
            "median_absolute_error": "{:.2f}",
            "RMSE": "{:.2f}",
            "within_10_eur": "{:.1%}",
            "within_20_eur": "{:.1%}",
            "within_50_eur": "{:.1%}",
        })
    )



Category: 10 best groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,fuel,181,6.44,2.32,11.03,78.5%,91.2%,100.0%
1,airbag,106,11.64,4.53,24.52,79.2%,90.6%,92.5%
2,electric / transmitter / databox / sensor,404,12.77,3.74,39.84,70.5%,86.9%,96.8%
3,brakes,214,13.39,5.52,34.50,79.9%,87.4%,95.3%
4,gear box / drive axle / middle axle,241,21.34,5.95,62.20,61.8%,71.4%,91.7%
5,vehicle exterior / suspension,294,34.84,3.64,150.41,82.3%,89.1%,91.8%
6,engine,249,140.37,3.49,496.86,67.5%,72.3%,84.3%


Category: 10 worst groups by MAE


,category,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
6,engine,249,140.37,3.49,496.86,67.5%,72.3%,84.3%
5,vehicle exterior / suspension,294,34.84,3.64,150.41,82.3%,89.1%,91.8%
4,gear box / drive axle / middle axle,241,21.34,5.95,62.20,61.8%,71.4%,91.7%
3,brakes,214,13.39,5.52,34.50,79.9%,87.4%,95.3%
2,electric / transmitter / databox / sensor,404,12.77,3.74,39.84,70.5%,86.9%,96.8%
1,airbag,106,11.64,4.53,24.52,79.2%,90.6%,92.5%
0,fuel,181,6.44,2.32,11.03,78.5%,91.2%,100.0%



Subcategory: 10 best groups by MAE


...

,subcategory,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
23,engine gasoline,21,1293.44,1369.50,1607.89,0.0%,0.0%,0.0%
22,adaptiv farthållare sensor,24,24.18,23.86,25.09,0.0%,50.0%,100.0%
21,contact roll airbag,20,17.43,8.92,29.67,60.0%,85.0%,85.0%
20,deliverer,23,14.71,2.39,22.46,56.5%,69.6%,100.0%
19,all,153,11.90,4.03,28.00,82.4%,85.6%,94.1%
18,right front,48,11.79,6.81,17.16,77.1%,77.1%,97.9%
17,actuator loom,20,11.33,1.21,22.48,65.0%,65.0%,95.0%
16,airbag control unit,20,10.93,7.61,15.15,65.0%,90.0%,100.0%
15,fuse box / electricity central,27,10.02,6.98,10.85,51.9%,100.0%,100.0%
14,passenger airbag,25,9.96,5.34,19.90,92.0%,92.0%,96.0%



Part name: 10 best groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
0,Distributors Vacuum regulator -,20,3.14,0.86,7.66,90.0%,90.0%,100.0%
1,Drive shaft -(Left front),32,5.09,2.84,9.01,90.6%,90.6%,100.0%
2,ABS Hydraulic pump -,24,5.37,5.32,5.69,100.0%,100.0%,100.0%
3,Trailing link rear -(Left),20,8.68,2.11,13.20,60.0%,95.0%,100.0%
4,Brake Caliper -(Left front),20,10.05,5.44,19.04,90.0%,90.0%,95.0%
5,Wheel bearing spindle shaft -(Left rear),21,11.42,3.32,30.37,81.0%,81.0%,95.2%
6,Drive shaft -(Right front),23,19.38,7.89,26.50,52.2%,56.5%,91.3%
7,Wheel bearing spindle shaft -(Right rear),20,21.11,4.76,46.84,85.0%,85.0%,85.0%


Part name: 10 worst groups by MAE


,part_name,count,MAE,median_absolute_error,RMSE,within_10_eur,within_20_eur,within_50_eur
7,Wheel bearing spindle shaft -(Right rear),20,21.11,4.76,46.84,85.0%,85.0%,85.0%
6,Drive shaft -(Right front),23,19.38,7.89,26.50,52.2%,56.5%,91.3%
5,Wheel bearing spindle shaft -(Left rear),21,11.42,3.32,30.37,81.0%,81.0%,95.2%
4,Brake Caliper -(Left front),20,10.05,5.44,19.04,90.0%,90.0%,95.0%
3,Trailing link rear -(Left),20,8.68,2.11,13.20,60.0%,95.0%,100.0%
2,ABS Hydraulic pump -,24,5.37,5.32,5.69,100.0%,100.0%,100.0%
1,Drive shaft -(Left front),32,5.09,2.84,9.01,90.6%,90.6%,100.0%
0,Distributors Vacuum regulator -,20,3.14,0.86,7.66,90.0%,90.0%,100.0%


## 29. Select The Best XGBoost Feature Set

Reuse the tuned native-categorical XGBoost hyperparameters across every feature-set experiment so the comparison stays focused on feature value instead of one-hot preprocessing noise.


In [191]:
excluded_features = xgboost_excluded_features

feature_sets = {
    "baseline only": features_baseline_only,
    "baseline + traficom_core": features_baseline_plus_traficom,
    "baseline + traficom_core + registry_lifecycle": (
        features_baseline_plus_traficom + registry_lifecycle_features
    ),
    "baseline + traficom_core + registry_lifecycle + traficom_extended": (
        features_baseline_plus_traficom
        + registry_lifecycle_features
        + traficom_extended_features
    ),
    "previous manual all-features set": manual_all_feature_groups,
    "all recommended features": recommended_model_features,
}

experiment_results = []
experiment_feature_details = []
experiment_predictions = {}

In [192]:
def prepare_experiment_features(feature_list, train_df, validation_df, excluded_features):
    requested_features = list(dict.fromkeys(feature_list))
    requested_features = [
        feature for feature in requested_features
        if feature not in excluded_features
    ]

    available_features = [
        feature for feature in requested_features
        if feature in train_df.columns
    ]
    missing_features = [
        feature for feature in requested_features
        if feature not in train_df.columns
    ]

    X_train_current = train_df[available_features].copy()
    X_validation_current = validation_df[available_features].copy()

    return (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    )


In [193]:
for model_name, feature_list in feature_sets.items():
    (
        requested_features,
        available_features,
        missing_features,
        X_train_current,
        X_validation_current,
    ) = prepare_experiment_features(
        feature_list,
        train_df,
        validation_df,
        excluded_features,
    )

    model_current, X_train_current_prepared, X_validation_current_prepared = fit_xgboost_model(
        X_train_current,
        y_train_log,
        X_validation_current,
        y_validation_log,
        selected_xgboost_params,
    )

    validation_pred_log_current = model_current.predict(X_validation_current_prepared)
    validation_pred_current = np.exp(validation_pred_log_current)
    experiment_predictions[model_name] = validation_pred_current

    validation_mae_current = mean_absolute_error(
        y_validation,
        validation_pred_current,
    )
    validation_rmse_current = np.sqrt(
        mean_squared_error(y_validation, validation_pred_current)
    )
    validation_r2_current = r2_score(
        y_validation,
        validation_pred_current,
    )

    experiment_results.append({
        "experiment": model_name,
        "raw_columns_requested": len(requested_features),
        "raw_columns_used": len(available_features),
        "raw_columns_missing": len(missing_features),
        "validation_MAE": validation_mae_current,
        "validation_RMSE": validation_rmse_current,
        "validation_R2": validation_r2_current,
    })

    experiment_feature_details.append({
        "experiment": model_name,
        "used_features_count": len(available_features),
        "used_features_list": available_features,
        "missing_features_list": missing_features,
    })


## 30. Validate The Best XGBoost Model
 MAE as the main validation metrics, backup R^2 and MSE, and extra MAPE. 


In [194]:
# Rebuild the best-model validation dataframe if this section is run on its own.
if "best_model_validation_df" not in globals():
    if "best_experiment_name" not in globals():
        if "experiment_results" not in globals() or len(experiment_results) == 0:
            raise NameError(
                "best_experiment_name is not defined. Run the feature-set comparison section first."
            )

        experiment_results_df = pd.DataFrame(experiment_results)
        best_experiment_name = experiment_results_df.sort_values(
            ["validation_MAE", "validation_RMSE"]
        ).iloc[0]["experiment"]

    best_validation_predictions = experiment_predictions[best_experiment_name]

    best_model_validation_df = pd.DataFrame({
        "actual_price": y_validation,
        "predicted_price": best_validation_predictions,
    })
    best_model_validation_df["residual"] = (
        best_model_validation_df["actual_price"]
        - best_model_validation_df["predicted_price"]
    )
    best_model_validation_df["absolute_error"] = (
        best_model_validation_df["residual"].abs()
    )
    best_model_validation_df["squared_error"] = (
        best_model_validation_df["residual"] ** 2
    )
    best_model_validation_df["absolute_percentage_error"] = (
        best_model_validation_df["absolute_error"]
        / best_model_validation_df["actual_price"].clip(lower=1)
    )
    best_model_validation_df["prediction_direction"] = np.where(
        best_model_validation_df["residual"] >= 0,
        "underpredicted",
        "overpredicted",
    )
    best_model_validation_df["actual_price_band"] = pd.qcut(
        best_model_validation_df["actual_price"],
        q=5,
        duplicates="drop",
    )

# Split the best-model validation errors by price band and prediction direction.
best_model_direction_summary = (
    best_model_validation_df
    .groupby(["actual_price_band", "prediction_direction"], observed=False)
    .agg(
        samples=("actual_price", "size"),
        mean_actual_price=("actual_price", "mean"),
        mean_predicted_price=("predicted_price", "mean"),
        mae=("absolute_error", "mean"),
    )
    .reset_index()
)

best_model_direction_summary.style.format({
    "mean_actual_price": "{:.1f}",
    "mean_predicted_price": "{:.1f}",
    "mae": "{:.2f}",
})

,actual_price_band,prediction_direction,samples,mean_actual_price,mean_predicted_price,mae
0,"(5.899, 47.4]",overpredicted,225,35.2,41.2,6.01
1,"(5.899, 47.4]",underpredicted,124,36.0,33.2,2.77
2,"(47.4, 82.6]",overpredicted,141,60.5,64.9,4.41
3,"(47.4, 82.6]",underpredicted,194,61.6,57.5,4.13
4,"(82.6, 154.06]",overpredicted,118,120.6,129.5,8.90
5,"(82.6, 154.06]",underpredicted,211,103.9,97.4,6.48
6,"(154.06, 248.42]",overpredicted,154,192.9,203.1,10.25
7,"(154.06, 248.42]",underpredicted,184,215.4,199.9,15.46
8,"(248.42, 4284.0]",overpredicted,141,767.5,962.5,195.00
9,"(248.42, 4284.0]",underpredicted,197,967.1,848.8,118.29


In [195]:
# Build a full best-model validation table for absolute and percentage error diagnostics.
best_validation_predictions = experiment_predictions[best_experiment_name]

best_model_validation_df = pd.DataFrame({
    "actual_price": y_validation,
    "predicted_price": best_validation_predictions,
})
best_model_validation_df["residual"] = (
    best_model_validation_df["actual_price"]
    - best_model_validation_df["predicted_price"]
)
best_model_validation_df["absolute_error"] = (
    best_model_validation_df["residual"].abs()
)
best_model_validation_df["squared_error"] = (
    best_model_validation_df["residual"] ** 2
)
best_model_validation_df["absolute_percentage_error"] = (
    best_model_validation_df["absolute_error"]
    / best_model_validation_df["actual_price"].clip(lower=1)
)
best_model_validation_df["prediction_direction"] = np.where(
    best_model_validation_df["residual"] >= 0,
    "underpredicted",
    "overpredicted",
)
best_model_validation_df["actual_price_band"] = pd.qcut(
    best_model_validation_df["actual_price"],
    q=5,
    duplicates="drop",
)

# The price-band summary includes MAPE because percentage error is informative across very different price levels.
best_model_price_band_summary = (
    best_model_validation_df
    .groupby("actual_price_band", observed=False)
    .agg(
        samples=("actual_price", "size"),
        min_actual_price=("actual_price", "min"),
        max_actual_price=("actual_price", "max"),
        mean_actual_price=("actual_price", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda s: np.sqrt(s.mean())),
        mape=("absolute_percentage_error", "mean"),
        median_residual=("residual", "median"),
    )
    .reset_index()
)

best_model_price_band_summary.style.format({
    "min_actual_price": "{:.1f}",
    "max_actual_price": "{:.1f}",
    "mean_actual_price": "{:.1f}",
    "mae": "{:.2f}",
    "rmse": "{:.2f}",
    "mape": "{:.1%}",
    "median_residual": "{:+.2f}",
})

,actual_price_band,samples,min_actual_price,max_actual_price,mean_actual_price,mae,rmse,mape,median_residual
0,"(5.899, 47.4]",349,5.9,47.4,35.5,5.00,7.68,16.9%,-1.39
1,"(47.4, 82.6]",335,47.6,82.6,61.1,4.24,6.13,6.9%,+0.92
2,"(82.6, 154.06]",329,82.8,153.9,109.9,10.11,13.85,9.2%,+2.93
3,"(154.06, 248.42]",338,154.1,248.3,205.1,15.21,25.35,7.6%,-0.58
4,"(248.42, 4284.0]",338,248.6,4284.0,883.9,53.25,98.13,7.5%,+6.62


In [196]:
experiment_results_df = pd.DataFrame(experiment_results)
baseline_row = experiment_results_df.loc[
    experiment_results_df["experiment"] == "baseline only"
].iloc[0]

experiment_results_df["delta_MAE_vs_baseline"] = (
    experiment_results_df["validation_MAE"]
    - baseline_row["validation_MAE"]
)
experiment_results_df["delta_RMSE_vs_baseline"] = (
    experiment_results_df["validation_RMSE"]
    - baseline_row["validation_RMSE"]
)
experiment_results_df["mae_rank"] = (
    experiment_results_df["validation_MAE"]
    .rank(method="dense")
    .astype(int)
)
experiment_results_df["rmse_rank"] = (
    experiment_results_df["validation_RMSE"]
    .rank(method="dense")
    .astype(int)
)

display_columns = [
    "experiment",
    "raw_columns_used",
    "validation_MAE",
    "delta_MAE_vs_baseline",
    "mae_rank",
    "validation_RMSE",
    "delta_RMSE_vs_baseline",
    "rmse_rank",
    "validation_R2",
]

best_experiment_name = experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
).iloc[0]["experiment"]


experiment_results_df.sort_values(
    ["validation_MAE", "validation_RMSE"]
)[display_columns].style.format({
    "validation_MAE": "{:.4f}",
    "delta_MAE_vs_baseline": "{:+.4f}",
    "validation_RMSE": "{:.4f}",
    "delta_RMSE_vs_baseline": "{:+.4f}",
    "validation_R2": "{:.4f}",
})

,experiment,raw_columns_used,validation_MAE,delta_MAE_vs_baseline,mae_rank,validation_RMSE,delta_RMSE_vs_baseline,rmse_rank,validation_R2
5,all recommended features,72,17.5436,-12.4143,1,45.9620,-36.7954,2,0.9934
4,previous manual all-features set,70,17.8580,-12.0999,2,45.6997,-37.0577,1,0.9935
1,baseline + traficom_core,28,27.6672,-2.2907,3,65.2942,-17.4631,3,0.9867
3,baseline + traficom_core + registry_lifecycle + traficom_extended,64,29.6573,-0.3006,4,83.0893,+0.3319,5,0.9785
0,baseline only,15,29.9579,+0.0000,5,82.7574,+0.0000,4,0.9787
2,baseline + traficom_core + registry_lifecycle,42,31.3370,+1.3791,6,102.2056,+19.4482,6,0.9675
